In [1]:
import json

In [2]:
class BusinessConfig:
    """
    Business unit economic assumptions 
    All percentages are in decimal form (0.5 = 50%) 
    """
    def __init__(
        self, 
        avg_lifetime: int = 36, 
        floor_months: int = 6, 
        margin_pct: float = 0.5,
        retention_success_rate: float = 0.2, 

         # NEW — real campaign economics
        contact_cost: float = 5.0,      # call/SMS/email cost
        incentive_cost: float = 20.0    # coupon/discount cost
        
    ): 
        if avg_lifetime < floor_months:
            raise ValueError("avg_lifetime must be >= floor_months")
        assert 0 <= margin_pct <= 1 
        assert contact_cost >= 0
        assert incentive_cost >= 0

        self.avg_lifetime = avg_lifetime
        self.floor_months = floor_months
        self.margin_pct = margin_pct
        
        # total intervention cost
        self.retention_cost = contact_cost + incentive_cost
        self.retention_success_rate = retention_success_rate

In [3]:
config = BusinessConfig()

In [4]:
#Economic Calculation Functions 
def calculate_base_remaining_value(tenure:int , monthly_charges: float, config: BusinessConfig):
    """
    Calculates future profit we expect from the customer
    if they don't churn.
    """
    
    if tenure < 0:
        raise ValueError("tenure cannot be negative")
    if monthly_charges < 0: 
        raise ValueError("monthly_charges cannot be negative")
    if monthly_charges == 0: 
        return 0.0, 0.0
    
    base_months = max(config.avg_lifetime - tenure, config.floor_months)
    monthly_margin = monthly_charges * config.margin_pct
    brv = base_months * monthly_margin

    return brv, monthly_margin

In [5]:
def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except:
        return default


def compute_customer_value(user_data, config):

    """ 
    Adapter: converts model input schema → business schema 
    """

    tenure = _safe_float(user_data.get("tenure"))
    monthly_charges = _safe_float(user_data.get("MonthlyCharges"))

    brv, monthly_margin = calculate_base_remaining_value(
        tenure,
        monthly_charges,
        config
    )

    return {
        "brv": round(brv, 2),
        "monthly_margin": round(monthly_margin, 2)
    }

In [6]:
#Expected Loss 
def calculate_expected_loss(p_churn, brv): 
    return p_churn * brv

In [7]:
def calculate_expected_gain(p_churn, brv, config):

    # Value we save IF customer was going to churn
    expected_recovered = brv * p_churn * config.retention_success_rate

    # Fixed intervention cost (VERY IMPORTANT FIX)
    retention_cost = config.retention_cost

    expected_gain = expected_recovered - retention_cost

    roi = expected_gain / retention_cost if retention_cost > 0 else 0

    return expected_gain, roi, retention_cost

In [8]:
# Decision Rule
def make_decision(expected_gain): 
    return "Retain" if expected_gain > 0 else "Let Go"

### What we can improve

1. we can make the data to be vectrorised so it can work faster

In [9]:
with open("segmentation_config.json", "r") as f:
    CONFIG = json.load(f)

median_brv = CONFIG["median_brv"]
churn_threshold = 0.314
def assign_segment(p_churn, brv):
    if brv >= median_brv and p_churn >= churn_threshold:
        return "High Value - High Risk"
    elif brv >= median_brv and p_churn < churn_threshold:
        return "High Value - Low Risk"
    elif brv < median_brv and p_churn >= churn_threshold:
        return "Low Value - High Risk"
    else:
        return "Low Value - Low Risk"

In [10]:
import math

def run_decision_engine(df, config):

    results = []

    for idx, row in df.iterrows():

        if pd.isna(row["tenure"]) or pd.isna(row["MonthlyCharges"]):
            raise ValueError(f"Invalid data at row {idx}")

        brv, monthly_margin = calculate_base_remaining_value(
            float(row["tenure"]),
            float(row["MonthlyCharges"]),
            config
        )

        expected_gain, roi, retention_cost = calculate_expected_gain(
            row["p_churn"],
            brv,
            config
        )

        decision = make_decision(expected_gain)

        results.append({
            "p_churn": row["p_churn"],
            "BRV": brv,
            "Expected_Gain": expected_gain,
            "ROI": roi,
            "Decision": decision,
            "Retention_cost": retention_cost
        })

    return pd.DataFrame(results)

In [11]:
def campaign_summary(df):
    targeted = df[df["Decision"] == "Retain"]

    total_cost = targeted["Retention_cost"].sum()
    total_gain = targeted["Expected_Gain"].sum()

    roi = total_gain / (total_cost + 1e-6)

    return {
        "Total Customers": len(df),
        "Targeted Customers": len(targeted),
        "Total Cost": total_cost,
        "Total Expected Gain": total_gain,
        "Campaign ROI": roi
    }

### TESTING

In [12]:
import joblib
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

model = joblib.load("churn_pipeline.pkl")

In [13]:
df["p_churn"] = model.predict_proba(df)[:, 1]

In [14]:
df["p_churn"].describe()

count    7043.000000
mean        0.264277
std         0.264652
min         0.000000
25%         0.030769
50%         0.134921
75%         0.437500
max         1.000000
Name: p_churn, dtype: float64

In [15]:
results_df = run_decision_engine(df, config)

# remove columns that will be regenerated
df = df.drop(columns=results_df.columns, errors="ignore")


In [16]:
df = pd.concat([df, results_df], axis=1)
df["Segment"] = df.apply(lambda x: assign_segment(x["p_churn"], x["BRV"]), axis=1)

In [17]:
# sanity check
# Check 1 — High p vs Low p Behavior 

df.sort_values("p_churn", ascending=False)[
    ["p_churn", "BRV", "Expected_Gain", "Decision"]
].head(10)

,p_churn,BRV,Expected_Gain,Decision
2191,1.0,1597.750,294.550,Retain
1371,1.0,1386.000,252.200,Retain
4326,1.0,1559.250,286.850,Retain
6089,1.0,1585.500,292.100,Retain
6894,1.0,1747.350,324.470,Retain
3749,1.0,1595.450,294.090,Retain
1704,1.0,1745.625,324.125,Retain
3159,1.0,1565.025,288.005,Retain
5783,1.0,1562.750,287.550,Retain
997,1.0,1508.800,276.760,Retain


In [18]:
df.sort_values("p_churn")[
    ["p_churn", "BRV", "Expected_Gain", "Decision"]
].head(10)

,p_churn,BRV,Expected_Gain,Decision
763,0.0,200.55,-25.0,Let Go
6063,0.0,57.60,-25.0,Let Go
5971,0.0,60.00,-25.0,Let Go
2176,0.0,58.20,-25.0,Let Go
4086,0.0,237.60,-25.0,Let Go
789,0.0,208.95,-25.0,Let Go
4920,0.0,60.30,-25.0,Let Go
5096,0.0,260.55,-25.0,Let Go
109,0.0,58.20,-25.0,Let Go
714,0.0,76.20,-25.0,Let Go


In [19]:
# Check 2 — High BRV Behavior 
df.sort_values("BRV", ascending=False)[
    ["p_churn", "BRV", "Expected_Gain", "Decision"]
].head(10)

,p_churn,BRV,Expected_Gain,Decision
2246,0.860000,1792.875,283.374500,Retain
678,0.642336,1781.175,203.822482,Retain
1719,0.687500,1781.175,219.911563,Retain
6482,1.000000,1775.375,330.075000,Retain
171,0.860000,1774.800,280.265600,Retain
2208,1.000000,1764.000,327.800000,Retain
4459,0.860000,1754.375,276.752500,Retain
6894,1.000000,1747.350,324.470000,Retain
1704,1.000000,1745.625,324.125000,Retain
3085,0.860000,1738.275,273.983300,Retain


In [20]:
# Check 3 — Distribution of Decisions 
df["Decision"].value_counts(normalize=True)

Decision
Let Go    0.670453
Retain    0.329547
Name: proportion, dtype: float64

In [21]:
# Check 4 — Campaign ROI 
summary = campaign_summary(df)
summary

{'Total Customers': 7043,
 'Targeted Customers': 2321,
 'Total Cost': np.float64(58025.0),
 'Total Expected Gain': np.float64(210284.63063232222),
 'Campaign ROI': np.float64(3.6240349957552467)}

In [22]:
# manual test case check 

test_cases = pd.DataFrame([
    {"tenure": 5, "MonthlyCharges": 100, "p_churn": 0.9},
    {"tenure": 30, "MonthlyCharges": 120, "p_churn": 0.7},
    {"tenure": 20, "MonthlyCharges": 40, "p_churn": 0.8},
    {"tenure": 35, "MonthlyCharges": 90, "p_churn": 0.1}
])

In [23]:
results_df = run_decision_engine(test_cases, config)

# remove columns that will be regenerated
test_cases = df.drop(columns=results_df.columns, errors="ignore")
tes_case = pd.concat([df, results_df], axis=1)
# sanity check
# Check 1 — High p vs Low p Behavior 

df.sort_values("p_churn", ascending=False)[
    ["p_churn", "BRV", "Expected_Gain", "Decision"]
].head()

,p_churn,BRV,Expected_Gain,Decision
2191,1.0,1597.75,294.55,Retain
1371,1.0,1386.00,252.20,Retain
4326,1.0,1559.25,286.85,Retain
6089,1.0,1585.50,292.10,Retain
6894,1.0,1747.35,324.47,Retain


In [24]:
df.groupby("Segment")["Decision"].value_counts()

Segment                 Decision
High Value - High Risk  Retain      2204
                        Let Go       459
High Value - Low Risk   Let Go      3802
                        Retain       117
Low Value - Low Risk    Let Go       461
Name: count, dtype: int64

In [25]:
df.groupby("Decision")["p_churn"].mean()

Decision
Let Go    0.104664
Retain    0.589004
Name: p_churn, dtype: float64

In [26]:
df.groupby("Decision")["BRV"].mean()

Decision
Let Go    247.967217
Retain    920.485954
Name: BRV, dtype: float64